In [0]:
DROP TABLE IF EXISTS nyc_taxi.bronze_green_trip;
DROP TABLE IF EXISTS nyc_taxi.bronze_payment;
DROP TABLE IF EXISTS nyc_taxi.bronze_type;
DROP TABLE IF EXISTS nyc_taxi.bronze_zone;

CREATE DATABASE IF NOT EXISTS nyc_taxi;

--payment
CREATE TABLE IF NOT EXISTS nyc_taxi.bronze_payment (
    payment_type_code STRING,
    payment_type STRING,
    file_name STRING,
    created_on TIMESTAMP
)
USING DELTA LOCATION 'abfss://nyc-taxi@gen2nyctaxi.dfs.core.windows.net/bronze/payment';


COPY INTO nyc_taxi.bronze_payment
FROM (
    SELECT payment_type_code,
        payment_type,
        _metadata.file_path AS file_name,
        CURRENT_TIMESTAMP() AS created_on
    FROM 'abfss://nyc-taxi@gen2nyctaxi.dfs.core.windows.net/landing/trip_payment'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true');

--zone

CREATE TABLE IF NOT EXISTS nyc_taxi.bronze_zone (
    LocationID STRING,
    Borough STRING,
    Zone STRING,
    service_zone STRING,
    file_name STRING,
    created_on TIMESTAMP
)
USING DELTA LOCATION 'abfss://nyc-taxi@gen2nyctaxi.dfs.core.windows.net/bronze/zone';

COPY INTO nyc_taxi.bronze_zone
FROM (
    SELECT
        LocationID,
        Borough,
        Zone,
        service_zone,
        _metadata.file_path AS file_name,
        CURRENT_TIMESTAMP() AS created_on
    FROM 'abfss://nyc-taxi@gen2nyctaxi.dfs.core.windows.net/landing/trip_zone'
)
FILEFORMAT = CSV
FORMAT_OPTIONS (
    'header' = 'true'
);

--type
CREATE TABLE IF NOT EXISTS nyc_taxi.bronze_type(
    trip_type STRING,
    description STRING,
    file_name STRING,
    created_on TIMESTAMP
) USING DELTA LOCATION 'abfss://nyc-taxi@gen2nyctaxi.dfs.core.windows.net/bronze/type';

COPY INTO nyc_taxi.bronze_type
FROM (
    SELECT 
        trip_type,
        description,
        _metadata.file_path AS file_name,
        CURRENT_TIMESTAMP() AS created_on
    FROM 'abfss://nyc-taxi@gen2nyctaxi.dfs.core.windows.net/landing/trip_type'
)
FILEFORMAT = CSV
FORMAT_OPTIONS (
   'header' = 'true'
);

--trip
CREATE TABLE IF NOT EXISTS nyc_taxi.bronze_green_trip (
    VendorID INT,
    lpep_pickup_datetime TIMESTAMP_NTZ,
    lpep_dropoff_datetime TIMESTAMP_NTZ,
    store_and_fwd_flag STRING,
    RatecodeID BIGINT,
    PULocationID INT,
    DOLocationID INT,
    passenger_count BIGINT,
    trip_distance DOUBLE,
    fare_amount DOUBLE,
    extra DOUBLE,
    mta_tax DOUBLE,
    tip_amount DOUBLE,
    tolls_amount DOUBLE,
    ehail_fee DOUBLE,
    improvement_surcharge DOUBLE,
    total_amount DOUBLE,
    payment_type BIGINT,
    trip_type BIGINT,
    congestion_surcharge DOUBLE,
    year INT,
    month INT,
    file_name STRING,
    created_on TIMESTAMP
)
USING DELTA LOCATION 'abfss://nyc-taxi@gen2nyctaxi.dfs.core.windows.net/bronze/green_taxi'
PARTITIONED BY (year, month)
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

COPY INTO nyc_taxi.bronze_green_trip
FROM (
SELECT 
    VendorID,
    lpep_pickup_datetime,
    lpep_dropoff_datetime,
    store_and_fwd_flag,
    RatecodeID,
    PULocationID,
    DOLocationID,
    passenger_count,
    trip_distance,
    fare_amount,
    extra,
    mta_tax,
    tip_amount,
    tolls_amount,
    ehail_fee,
    improvement_surcharge,
    total_amount,
    payment_type,
    trip_type,
    congestion_surcharge,
    year,
    month,
    _metadata.file_path AS file_name,
    CURRENT_TIMESTAMP() AS created_on
FROM 'abfss://nyc-taxi@gen2nyctaxi.dfs.core.windows.net/landing/green_taxi'
)
FILEFORMAT = parquet;

